# Private full OPLoRA16 training
Fresh independent run from the identical original Qwen3 checkpoint. Seed6304,rank16,alpha32,protection128,dropout.05,1250updates/40000examples,microbatch2accumulation16; unchanged manual recipe. Attach original private Task1 training input. Keep notebook private. Previous jobs must be terminal. Launch once with Save Version > Save & Run All. Exact FP32 SVD preprocessing may take time before optimizer updates. All1250 updates requested with30000s safety bound inside the Kaggle session limit. Preserve and verify saved output archive; a partial run is not completion.


In [ ]:
# Set only after the previous Kaggle draft is stopped and no training is active.
BATCH_HANDOFF_CONFIRMED = False
assert BATCH_HANDOFF_CONFIRMED, 'Stop the prior draft/session, then confirm handoff before Save & Run All'


In [ ]:
import subprocess, sys, pathlib, json, zipfile, urllib.request
REPO_COMMIT = "cf7f042114a0a924403ed55ef9ee5982946ecf38"
work = pathlib.Path("/kaggle/working")
source = work / "pa3-source"
if not source.exists():
    archive = work / "pa3-source.zip"
    urllib.request.urlretrieve(f"https://github.com/sohauv/atml-pa3/archive/{REPO_COMMIT}.zip", archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall(work)
    (work / f"atml-pa3-{REPO_COMMIT}").rename(source)
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0", "transformers==4.51.3", "accelerate==1.6.0", "datasets==3.5.0", "numpy==2.2.5"])
print("Pinned source prepared:", REPO_COMMIT)

subprocess.check_call([sys.executable, '-c', "import torch; assert torch.cuda.is_available(), 'Select free T4 GPU'; print('VERIFIED_GPU', torch.__version__, torch.cuda.get_device_name(0), flush=True)"])


In [ ]:
# Verify all original input bytes; tolerate only byte-identical duplicate copies.
import sys, pathlib, zipfile, hashlib
sys.path.insert(0, str(source))
from scripts.kaggle_task1 import unpack_inputs, select_original_input
EXPECTED_TRAIN_SHA = '644a46f9ab7a8d368499b2054229cc7324614d97596fbdee1ac3ab5cbb584de9'
archives = list(pathlib.Path('/kaggle/input').rglob('pa3-task1-input.zip'))
if len(archives) == 1:
    input_archive = archives[0]
else:
    original = select_original_input('/kaggle/input', EXPECTED_TRAIN_SHA)
    input_archive = work / 'pa3-task1-input.zip'
    if not input_archive.exists():
        with zipfile.ZipFile(input_archive, 'x', zipfile.ZIP_DEFLATED) as z:
            for name in ('train.jsonl', 'metadata.json', 'source_ids.json'):
                z.write(original / name, 'task1/' + name)
data_dir = unpack_inputs(input_archive, work / 'pa3/data')
assert hashlib.file_digest((data_dir / 'train.jsonl').open('rb'), 'sha256').hexdigest() == EXPECTED_TRAIN_SHA
print('Original training data checksum verified:', data_dir, flush=True)


In [ ]:
import pathlib, subprocess, sys, json, zipfile, hashlib
run_id = 'oplora16-kaggle-20261101'
run = work / 'pa3/runs' / run_id
assert not run.exists(), 'Independent OPLoRA run must start in a fresh directory'
command = [sys.executable, '-m', 'scripts.kaggle_task1', '--archive', str(input_archive), '--method', 'oplora16', '--run-id', run_id, '--seconds', '30000', '--max-updates', '1250', '--handoff-verified']
print('START_FULL_OPLORA16', 'original checkpoint; rank16 alpha32 protection128 seed6304; target1250', flush=True)
process = subprocess.run(command, cwd=source)
status = json.loads((run / 'status.json').read_text()) if (run / 'status.json').is_file() else None
complete = []
if run.exists():
    for marker in run.rglob('complete.json'):
        cp = marker.parent
        if all((cp / name).is_file() for name in ('summary.json','recovery.pt','projected_factors.pt')):
            summary = json.loads((cp / 'summary.json').read_text())
            complete.append((summary['update'], cp, summary))
latest = max(complete, key=lambda row: row[0]) if complete else None
step = latest[0] if latest else 0
summary = latest[2] if latest else None
finished = bool(process.returncode == 0 and status and status.get('finished') and summary and summary['update'] == 1250 and summary['successful_optimizer_updates'] == 1250 and summary['skipped_updates'] == 0)
evidence = {'returncode': process.returncode, 'status': status, 'summary': summary, 'recoverable_checkpoint': str(latest[1]) if latest else None, 'full_training_verified': finished, 'required_markers_present': {str(n): (run / ('step-%04d' % n) / 'complete.json').is_file() for n in (312,625,937,1250)}}
manifest = work / 'oplora16-backup-evidence.json'
manifest.write_text(json.dumps(evidence, indent=2))
archive = work / ('oplora16-backup-step-%04d.zip' % step)
with zipfile.ZipFile(archive, 'x', zipfile.ZIP_DEFLATED) as z:
    if run.exists():
        for file in run.rglob('*'):
            if file.is_file(): z.write(file, str(pathlib.Path('oplora16-recovery') / file.relative_to(run)))
    z.write(manifest, manifest.name)
with zipfile.ZipFile(archive) as z: assert z.testzip() is None
evidence.update(archive=str(archive), archive_bytes=archive.stat().st_size, archive_sha256=hashlib.file_digest(archive.open('rb'),'sha256').hexdigest())
manifest.write_text(json.dumps(evidence, indent=2))
print('DOWNLOAD_AND_VERIFY_OPLORA16', json.dumps(evidence), flush=True)
if not finished: print('INCOMPLETE_RUN: inspect actual failure or bounded checkpoint; do not claim1250 completion', flush=True)
